In [1]:
import gymnasium as gym
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
import torch

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
device = torch.device("cpu")
device

device(type='cpu')

create multiple parallel environments to improve data collection efficiency for training:

In [4]:
env = env = make_vec_env("Pendulum-v1", n_envs=4)

In [5]:
model = A2C("MlpPolicy", env, verbose=1, device="cpu")
model.learn(total_timesteps=1_000_000)

Using cpu device
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 200       |
|    ep_rew_mean        | -1.01e+03 |
| time/                 |           |
|    fps                | 18177     |
|    iterations         | 100       |
|    time_elapsed       | 0         |
|    total_timesteps    | 2000      |
| train/                |           |
|    entropy_loss       | -1.43     |
|    explained_variance | 0.0375    |
|    learning_rate      | 0.0007    |
|    n_updates          | 99        |
|    policy_loss        | -45.1     |
|    std                | 1.01      |
|    value_loss         | 752       |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 200       |
|    ep_rew_mean        | -1.18e+03 |
| time/                 |           |
|    fps                | 18061     |
|    iterations         | 200       |
|    time_elapsed       | 0      

In [6]:
test_env = gym.make("Pendulum-v1")

num_eval_episodes = 100
episode_rewards = []

for episode in range(num_eval_episodes):
    state, _ = test_env.reset()
    done = False
    total_reward = 0.0

    while not done:
        action, _ = model.predict(state, deterministic=True)
        state, reward, terminated, truncated, _ = test_env.step(action)
        done = terminated or truncated

        total_reward += reward

    episode_rewards.append(total_reward)

test_env.close()

average_reward = sum(episode_rewards) / num_eval_episodes
print(f"Average reward over {num_eval_episodes} episodes: {average_reward:.2f}")

Average reward over 100 episodes: -244.09
